# 3. 从教学模块映射到 icefall Zipformer2

**研究问题**：Zipformer2 如何在不同时间分辨率处理序列，并减少重复计算、保留旁路信息？以下对照 [icefall LibriSpeech Zipformer2 源码](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py)，读取版本为提交 `f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1`（2026-11-14 核对）。源码链接固定到该提交；本 notebook 只用 PyTorch 实现可运行的简化实验，不安装 icefall。

本仓库对应练习：`zipformer_resample`、`zipformer_shared_attention`、`zipformer_bypass`；先修：`conformer-core`。这些练习刻意省略了完整 Zipformer2 的位置分数、非线性注意力、调度、mask、卷积及其余投影。

本 notebook 按[官方中文介绍的“方法”第 1–6 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#方法)组织。文章解释 Zipformer 设计；源码链接固定到 icefall Zipformer2 的提交 f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1。概念和固定版本实现的差别在各节标明。


## 整体架构图




In [ ]:
import math
import torch
from torch.nn import functional as F

torch.manual_seed(13)
B,T,D=2,11,8
x=torch.arange(B*T*D,dtype=torch.float32).reshape(B,T,D)/100

## 1. 多时间分辨率

icefall 的 [`Zipformer2`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L49) 包含多个 encoder stack；[`DownsampledZipformer2Encoder`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1127) 将某个 stack 放在较低帧率运行，再上采样并与原分辨率输入组合。示意：`[B,T,D] → [B,ceil(T/r),D] → encoder → [B,T,D]`。**注意源码内部常用 `[T,B,D]`**，下表和实验为教学统一写成 `[B,T,D]`。

简化实验用每 $r$ 帧平均 $z_u=\frac1{n_u}\sum_{t\in G_u}x_t$，末组只除以有效帧数，再用重复采样恢复长度。icefall 的 `SimpleDownsample` 用可学习的组内位置权重，对末组重复最后一帧后加权；`SimpleUpsample` 重复低帧，再由 `BypassModule` 与原输入组合。本实验的等权平均、末组只计有效帧，均是教学简化。

In [ ]:
def simple_resample(x,factor):
    b,t,d=x.shape
    groups=[]
    for start in range(0,t,factor): groups.append(x[:,start:start+factor].mean(dim=1))
    low=torch.stack(groups,dim=1)
    restored=low.repeat_interleave(factor,dim=1)[:,:t]
    return low,restored

for r in (1,2,4):
    low,restored=simple_resample(x,r)
    print(f'r={r}: {tuple(x.shape)} -> {tuple(low.shape)} -> {tuple(restored.shape)}')
    assert low.shape==(B,math.ceil(T/r),D) and restored.shape==x.shape
assert torch.allclose(simple_resample(x,4)[0][:,-1],x[:,-3:].mean(1))

### 1.1 六个 stack 与学习型降采样

[文章第 1 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#1-downsampled-encoder-structure)的示例先把 100 Hz 输入降到 50 Hz，六个 stack 的内部帧率依次为 50、25、12.5、6.25、12.5、25 Hz，对应相对 50 Hz 的因子 (1,2,4,8,4,2)。降采样 stack 在上采样和 bypass 后回到 50 Hz，因此下一 stack 的输入仍在 50 Hz。不同 stack 也可有不同通道数，源码在 stack 间对齐宽度。

源码 SimpleDownsample 将每组 r 帧按 softmax(bias) 得到的 r 个权重加权求和，末组不满时重复最后一帧；SimpleUpsample 重复低帧，再经 BypassModule 与原输入混合。[源码](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1218-L1290)。例如 r=2 时，z_u=a_0 x_{2u}+a_1 x_{2u+1} 且 a_0+a_1=1。前面的 simple_resample 用等权平均且末组只计有效帧，是更简化的练习。

In [ ]:
def learned_group_downsample(x, logits, factor):
    # [B,T,D] -> [B,ceil(T/factor),D]
    b,t,d=x.shape
    pad=(-t)%factor
    if pad: x=torch.cat([x,x[:,-1:].expand(b,pad,d)],dim=1)
    groups=x.reshape(b,-1,factor,d)
    return (groups*logits.softmax(0)[None,None,:,None]).sum(2)

for stack,factor in enumerate((1,2,4,8,4,2),1):
    low=learned_group_downsample(x,torch.zeros(factor),factor)
    restored=low.repeat_interleave(factor,dim=1)[:,:T]
    print(stack, factor, tuple(x.shape), tuple(low.shape), tuple(restored.shape))
    assert low.shape==(B,math.ceil(T/factor),D) and restored.shape==x.shape
logits=torch.tensor([0.,math.log(3.)],requires_grad=True)
low=learned_group_downsample(x[:,:3],logits,2)
assert torch.allclose(low[:,0],0.25*x[:,0]+0.75*x[:,1])
assert torch.allclose(low[:,1],x[:,2])
low.sum().backward()
assert logits.grad is not None and torch.isfinite(logits.grad).all()

## 2. 注意力权重共享

[`Zipformer2EncoderLayer`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L511) 使用 [`RelPositionMultiheadAttentionWeights`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1398) 生成注意力权重，再供不同路径使用。教学公式：$A=\mathrm{softmax}(QK^\top/\sqrt{d_k})$，$Y_1=AV_1$，$Y_2=AV_2$。`Q,K:[B,T,d_k]`，`A:[B,T,T]`，`V_i:[B,T,d_i]`，`Y_i:[B,T,d_i]`。只计算一次 $A$ 并复用；实际源码还有位置分数、mask、非线性注意力和其他投影。

### 2.1 Zipformer block 的计算顺序

[文章第 2 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#2-zipformer-block)的核心是一次 MHAW 权重计算供 NLA 和两次 SA 复用。固定版本源码的主路径为：

输入 → MHAW(A) → FFN1 → NLA(A 的第 1 头) → SA1(A) → Conv1 → FFN2 → 中间 Bypass → SA2(A) → Conv2 → FFN3 → BiasNorm → 末尾 Bypass。

MHAW 权重形状 [H,B,T,T]，残差路径保持 [T,B,D]。Conv1/2 使用线性投影、GLU 和时间深度卷积，并加入 Balancer、Whiten、SwooshR 与因果分块选项。上面只列主要前向路径，省略训练时随机跳过模块等细节。[Zipformer2EncoderLayer 源码](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L511-L840)。

In [ ]:
def shared_attention(q,k,v1,v2):
    weights=torch.softmax(q@k.transpose(-2,-1)/math.sqrt(q.shape[-1]),dim=-1)
    return weights,weights@v1,weights@v2

q=torch.randn(B,5,4,requires_grad=True); k=torch.randn(B,5,4,requires_grad=True)
v1=torch.randn(B,5,3,requires_grad=True); v2=torch.randn(B,5,6,requires_grad=True)
w,y1,y2=shared_attention(q,k,v1,v2)
print('Q/K:',q.shape,'A:',w.shape,'V1/Y1:',v1.shape,y1.shape,'V2/Y2:',v2.shape,y2.shape)
assert w.shape==(B,5,5) and y1.shape==v1.shape and y2.shape==v2.shape
assert torch.allclose(w.sum(-1),torch.ones(B,5))
(y1.square().sum()+y2.square().sum()).backward()
assert all(t.grad is not None and torch.isfinite(t.grad).all() for t in (q,k,v1,v2))

### 2.2 非线性注意力：共享权重，改变被聚合的内容

**原理。** 源码的 [`Zipformer2EncoderLayer`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L511) 先用 Q、K 及相对位置计算 `attn_weights:[H,B,T,T]`；[`NonlinAttention`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1896) 只取第一个头的权重 `A:[B,T,T]`。同一层的两次 [`SelfAttention`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1739) 则使用完整的多头权重。权重矩阵 A 已经是 softmax 之后的结果，**不是 Q/K/V 本身**。

设输入 `u:[B,T,D]`，隐藏维 `C`，线性投影并拆成三路 `s,x,y:[B,T,C]`。教学版省略源码中的 Balancer、Whiten、ScaledLinear 等训练相关模块，保留其核心数据流：

$$
(s,x,y)=\operatorname{split}(W_{in}u),\quad
g=x\odot\tanh(s),\quad
m=Ag,\quad
z=W_{out}(m\odot y).
$$

`g:[B,T,C]`，`A:[B,T,T]`，`m=A@g:[B,T,C]`，`m*y:[B,T,C]`，最终 `z:[B,T,D]`。自注意力分支主要计算 `A@V` 后做输出投影；非线性分支在时间聚合前后各有一次输入相关的乘法。源码采用 `[T,B,D]` 和 `[H,B,T,T]` 布局；下面沿用本 notebook 的 `[B,T,D]` 方便观察。

In [ ]:
from torch import nn

class TeachingNonlinAttention(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.in_proj = nn.Linear(dim, 3 * hidden)
        self.out_proj = nn.Linear(hidden, dim)

    def forward(self, inputs, weights, trace=False):
        batch, frames, dim = inputs.shape
        assert weights.shape == (batch, frames, frames)
        s, content, gate = self.in_proj(inputs).chunk(3, dim=-1)
        gated_content = content * torch.tanh(s)
        mixed = weights @ gated_content
        output = self.out_proj(mixed * gate)
        if trace:
            shapes = {name: tuple(tensor.shape) for name, tensor in (
                ('input', inputs), ('s/content/gate', s),
                ('shared_weights', weights), ('gated_content', gated_content),
                ('time_mixed', mixed), ('output', output))}
            return output, shapes
        return output

nonlin = TeachingNonlinAttention(D, hidden=6)
inputs = torch.randn(B, 5, D, requires_grad=True)
# 上一节的 w 来自同一组 Q/K；这里直接复用，并不重新计算注意力分数。
shared_w = w.detach()  # 上一节已做反向传播；这里只复用其数值
nonlin_out, nonlin_shapes = nonlin(inputs, shared_w, trace=True)
print(nonlin_shapes)
assert nonlin_out.shape == inputs.shape

# 用独立展开式核对运算顺序，也确认第一头权重在教学版中的形状。
s, content, gate = nonlin.in_proj(inputs).chunk(3, dim=-1)
expected = nonlin.out_proj((shared_w @ (content * torch.tanh(s))) * gate)
assert torch.allclose(nonlin_out, expected, atol=1e-6)
nonlin_out.square().mean().backward()
assert inputs.grad is not None and torch.isfinite(inputs.grad).all()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in nonlin.parameters())
print('非线性注意力形状、公式与梯度检查通过')

### 2.3 逐通道 bypass

源码 [`BypassModule`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1073) 将输入与模块输出按通道组合。教学公式 $z=(1-c)\odot x+c\odot y$，`x,y:[B,T,D]`、`c:[D]` 经广播为 `[B,T,D]`。$c_d=0$ 选旁路，$c_d=1$ 选新输出。源码还涉及训练时按序列跳过模块、直通和参数约束等行为，这里只验证混合的数值性质。

In [ ]:
def bypass(x,y,c): return (1-c)*x+c*y

before=torch.randn(B,T,D,requires_grad=True)
after=torch.randn(B,T,D,requires_grad=True)
c=torch.linspace(0,1,D,requires_grad=True)
z=bypass(before,after,c)
print('输入/候选/通道权重/输出:',before.shape,after.shape,c.shape,z.shape)
assert z.shape==(B,T,D)
assert torch.allclose(z[...,0],before[...,0]) and torch.allclose(z[...,-1],after[...,-1])
z.square().sum().backward()
assert all(v.grad is not None and torch.isfinite(v.grad).all() for v in (before,after,c))

## 3. BiasNorm

**问题**：[文章第 3 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#3-biasnorm)指出 LayerNorm 的减均值和按方差缩放可能造成训练中的幅度问题。BiasNorm 用逐通道可学习偏置 b:[D] 参与分母，用标量 gamma 控制输出尺度：

$$y=\frac{x}{\sqrt{D^{-1}\sum_d(x_d-b_d)^2+\epsilon}}\exp(\gamma).$$

输入与输出 [B,T,D]，分母 [B,T,1]。分子仍是 x，且没有减均值。下面仅实现公式；源码另有自定义反向和参数限制。[BiasNorm 源码](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/scaling.py#L391-L448)。

In [ ]:
class TeachingBiasNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__()
        self.bias=nn.Parameter(torch.zeros(dim))
        self.log_scale=nn.Parameter(torch.zeros(()))
        self.eps=eps
    def forward(self,x):
        denominator=((x-self.bias).square().mean(-1,keepdim=True)+self.eps).sqrt()
        return x/denominator*self.log_scale.exp()

bias_norm=TeachingBiasNorm(D)
bx=torch.randn(B,5,D,requires_grad=True)
by=bias_norm(bx)
denominator=((bx-bias_norm.bias).square().mean(-1,keepdim=True)+bias_norm.eps).sqrt()
print('input',bx.shape,'denominator',denominator.shape,'output',by.shape)
assert by.shape==bx.shape and denominator.shape==(B,5,1)
assert torch.allclose(F.layer_norm(bx+2,(D,)),F.layer_norm(bx,(D,)),atol=1e-6)
assert not torch.allclose(bias_norm(bx+2),by)
by.square().mean().backward()
assert bx.grad is not None and bias_norm.bias.grad is not None

## 4. Swoosh 激活函数

[文章第 4 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#4-swoosh-激活函数)给出的逐元素公式如下；输入和输出形状相同：

$$\mathrm{Swish}(x)=x\sigma(x),$$
$$\mathrm{SwooshR}(x)=\log(1+e^{x-1})-0.08x-0.313261687,$$
$$\mathrm{SwooshL}(x)=\log(1+e^{x-4})-0.08x-0.035.$$

这里用 softplus 稳定计算对数指数项。R/L 的平移位置不同，负侧保留渐近斜率 -0.08。文章将 SwooshL 用于倾向于“通常关闭”的模块，将 SwooshR 用于卷积等模块；以下数值实验不检验识别质量。

In [ ]:
def swoosh_r(v): return F.softplus(v-1)-0.08*v-0.313261687
def swoosh_l(v): return F.softplus(v-4)-0.08*v-0.035
points=torch.tensor([-10.,-2.,0.,2.,6.],requires_grad=True)
for name,fn in [('Swish',lambda v:v*torch.sigmoid(v)),('SwooshR',swoosh_r),('SwooshL',swoosh_l)]:
    values=fn(points)
    slopes=torch.autograd.grad(values.sum(),points,retain_graph=True)[0]
    print(name,'value',values.detach().round(decimals=4).tolist(),'slope',slopes.detach().round(decimals=4).tolist())
    assert values.shape==points.shape and torch.isfinite(values).all() and torch.isfinite(slopes).all()
assert abs(swoosh_r(torch.tensor(0.)).item())<1e-6

## 5. ScaledAdam、Eden 与同形状参数分组

[文章第 5 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#5-scaledadam)讨论两个更新方向：令 r=RMS(theta)，在 Adam 型方向更新中乘 r，使相对更新幅度更接近；再通过沿 theta 本身的方向显式学习参数尺度。方向项的教学式：

$$\Delta_{\rm dir}=-\alpha r\frac{\hat m}{\sqrt{\hat v}+\epsilon}.$$

参数尺度方向的教学式为

$$\Delta_{\rm scale}=-\eta\alpha\theta\frac{\hat n}{\sqrt{\hat w}+\epsilon},$$

其中 $\hat n,\hat w$ 来自参数尺度梯度的动量，在这一简化例子中均为标量。theta、m、v 与更新均为 [D]，r 为标量。完整 ScaledAdam 还维护参数尺度的动量、裁剪及其他状态；下面只验证这个方向项的尺度关系，不能当作完整优化器使用。[ScaledAdam 源码](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/optim.py#L234)。

Eden 的教学倍率为

$$E(t,e)=\left(\frac{t^2+T_s^2}{T_s^2}\right)^{-1/4}
\left(\frac{e^2+T_e^2}{T_e^2}\right)^{-1/4}
\operatorname{warmup}(t).$$

其中 $t$ 为 step，$e$ 为 epoch，$T_s,T_e$ 控制衰减起点。Eden 同时依赖 step 和 epoch；相同 shape 的参数可合并为批次更新。以下演示公式和分组，不测吞吐。

In [ ]:
def scaled_direction(theta,mhat,vhat,lr):
    return -lr*theta.square().mean().sqrt()*mhat/vhat.sqrt()
theta=torch.tensor([0.5,-1.,2.])
mhat=torch.tensor([0.1,-0.2,0.3])
vhat=torch.tensor([0.04,0.09,0.16])
delta=scaled_direction(theta,mhat,vhat,0.01)
assert delta.shape==theta.shape
assert torch.allclose(scaled_direction(10*theta,mhat,vhat,0.01),10*delta)
scale_delta=-0.1*0.01*theta*0.2/(0.04**0.5)  # η、α、n、w 的教学数值
assert scale_delta.shape==theta.shape and torch.allclose(scale_delta/(theta+1e-12),torch.full_like(theta,-0.001))
print('theta RMS',theta.square().mean().sqrt().item(),'direction',delta.tolist(),'scale',scale_delta.tolist())

def eden_factor(step,epoch,step_scale,epoch_scale,start=0.5,warmup=10):
    a=((step*step+step_scale*step_scale)/(step_scale*step_scale))**(-0.25)
    b=((epoch*epoch+epoch_scale*epoch_scale)/(epoch_scale*epoch_scale))**(-0.25)
    return a*b*(start+(1-start)*min(step/warmup,1.))
print('Eden factor',[(s,round(eden_factor(s,2,100,10),4)) for s in (0,5,10,100,300)])
groups={}
for name,tensor in [('a',torch.zeros(2,3)),('b',torch.ones(2,3)),('c',torch.zeros(4))]:
    groups.setdefault(tuple(tensor.shape),[]).append(name)
print('shape groups',groups)
assert groups[(2,3)]==['a','b']

## 6. 激活值限制：Balancer 与 Whitener

[文章第 6 节](https://k2-fsa.org/zh-CN/blog/2023/12/15/zipformer-details/#6-激活值限制)针对两类训练问题：单通道的正值比例或幅度异常，以及通道之间高度相关。Balancer 关注每通道的正值比例与幅度；Whitener 关注中心化协方差 C 的特征值分布。对每个通道，源码将正值比例的上下界近似转为 $\mu/\sigma$ 的上下界，并将平均绝对值的上下界近似转为 RMS 上下界。教学惩罚可写作 $P_c=|m_c-\operatorname{clip}(m_c,m_{\min},m_{\max})|+|r_c-\operatorname{clip}(r_c,r_{\min},r_{\max})|$，其中 $m_c=\mu_c/\sigma_c$，$r_c=\operatorname{RMS}(x_c)$。

Whitener 令 $C=X_c^T X_c:[D,D]$，文章指标为：

$$M=\frac{\sum_{ij} C_{ij}^2/D}{(\mathrm{tr}(C)/D)^2}.$$

理想各向同性协方差接近 1，少数方向占主导时指标变大。两者在源码的前向是恒等映射，反向按概率加入约束梯度；并不等于 BatchNorm。下方用 x+lambda(P(x)-stopgrad(P(x))) 仅演示“前向不变、反向增添梯度”，省略阈值转换、采样、梯度缩放和内存优化。[Balancer](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/scaling.py#L677-L844)；[Whiten](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/scaling.py#L920-L990)。

In [ ]:
def channel_penalty(x):
    mean=x.mean(0)
    std=x.var(0,unbiased=False).add(1e-8).sqrt()
    rms=x.square().mean(0).add(1e-8).sqrt()
    ratio=mean/std
    return ((ratio-ratio.clamp(-0.5,0.5)).abs()
            +(rms-rms.clamp(0.5,2.0)).abs()).sum()
def whitening_metric(x):
    centered=x-x.mean(0,keepdim=True)
    cov=centered.T@centered
    d=x.shape[1]
    return (cov.square().sum()/d)/(cov.diagonal().mean().square()+1e-8)
torch.manual_seed(17)
raw=torch.randn(64,4)
raw[:,1]=raw[:,0]+0.02*torch.randn(64)
raw[:,2]+=4.0  # 让一个通道的正值比例和 RMS 超出教学阈值
bad=raw.detach().clone().requires_grad_()
penalty=channel_penalty(bad)+whitening_metric(bad)
out=bad+0.01*(penalty-penalty.detach())
print('input/output',bad.shape,out.shape,'channel penalty',round(channel_penalty(bad).item(),3),'whitening metric',round(whitening_metric(bad).item(),3))
assert torch.equal(out,bad) and channel_penalty(bad)>0
out.square().sum().backward()
assert torch.isfinite(bad.grad).all() and not torch.allclose(bad.grad,2*bad.detach())
independent=torch.randn(64,4)
assert whitening_metric(raw)>whitening_metric(independent)

## 7. 源码、练习与本 notebook 的对应关系

| 学习概念 | 本仓库题目 | icefall 源码对象 | 本本实验省略 |
|---|---|---|---|
| 多时间分辨率 | `zipformer_resample` | [`Zipformer2` / `DownsampledZipformer2Encoder`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1127) | 实际 encoder stack、宽度对齐、训练配置 |
| 权重共享 | `zipformer_shared_attention` | [`RelPositionMultiheadAttentionWeights` / `Zipformer2EncoderLayer`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1398) | 相对位置、mask、非线性注意力 |
| 非线性注意力 | 本 notebook 新增实验 | [`NonlinAttention`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1896) | Balancer、Whiten、ScaledLinear、训练时权重扰动 |
| BiasNorm / Swoosh | 第 3–4 节 | scaling.py | 源码反向和参数约束 |
| ScaledAdam / Eden | 第 5 节 | optim.py | 完整优化器状态和训练 |
| Balancer / Whitener | 第 6 节 | scaling.py | 阈值转换、采样和梯度策略 |
| 逐通道 bypass | `zipformer_bypass` | [`BypassModule`](https://github.com/k2-fsa/icefall/blob/f8c8578c9dbfa1183cc949b89b5d21bd33aa2ce1/egs/librispeech/ASR/zipformer/zipformer.py#L1073) | 参数约束与训练调度 |

**验证依据与适用边界**：上述断言仅检验简化公式、形状和梯度。不能由此推断 Zipformer2 的实际参数量、识别质量、吞吐或流式延迟。若要研究真实行为，需另行固定 icefall 版本、配置、输入和运行环境。